In [ ]:
from utils import *
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from datasets import SRDataset
from easydict import EasyDict as edict

In [ ]:
# Data parameters
csv_folder = "data"  # folder with CSV data files
test_data_names = ["Set5", "Set14", "B100", "Urban100", "valid"]
HR_data_folders = [
    "data/benchmark/Set5/HR",
    "data/benchmark/Set14/HR",
    "data/benchmark/B100/HR",
    "data/benchmark/Urban100/HR",
    "data/DIV2K_valid_HR",
]
LR_data_folders = [
    "data/benchmark/Set5/LR_bicubic/X4",
    "data/benchmark/Set14/LR_bicubic/X4",
    "data/benchmark/B100/LR_bicubic/X4",
    "data/benchmark/Urban100/LR_bicubic/X4",
    "data/DIV2K_valid_LR_bicubic/X4",
]
srgan_checkpoint = "./checkpoints/checkpoint_srgan_best.pth.tar"
scaling_factor = 4
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
srgan_generator = torch.load(srgan_checkpoint, weights_only=False)["generator"].to(
    device
)
srgan_generator.eval()
model = srgan_generator

In [ ]:
# Evaluate
for i in range(len(test_data_names)):
    print("\nFor %s:\n" % test_data_names[i])
    # Custom dataloader
    config = edict()
    config.csv_folder = csv_folder
    config.HR_data_folder = HR_data_folders[i]
    config.LR_data_folder = LR_data_folders[i]
    config.crop_size = 0
    config.scaling_factor = scaling_factor
    test_dataset = SRDataset(split=test_data_names[i], config=config)
    test_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True
    )
    PSNRs = AverageMeter()
    SSIMs = AverageMeter()
    with torch.no_grad():
        for i, (lr_imgs, hr_imgs) in enumerate(test_loader):
            lr_imgs = lr_imgs.to(device)
            hr_imgs = hr_imgs.to(device)
            lr_imgs = convert_image(
                lr_imgs, source="[0, 1]", target="imagenet-norm", device=device
            )
            hr_imgs = convert_image(
                hr_imgs, source="[0, 1]", target="[-1, 1]", device=device
            )

            sr_imgs = model(lr_imgs)

            sr_imgs_y = convert_image(
                sr_imgs, source="[-1, 1]", target="y-channel", device=device
            ).squeeze(0)
            hr_imgs_y = convert_image(
                hr_imgs, source="[-1, 1]", target="y-channel", device=device
            ).squeeze(0)
            psnr = peak_signal_noise_ratio(
                hr_imgs_y.cpu().numpy(), sr_imgs_y.cpu().numpy(), data_range=255.0
            )
            ssim = structural_similarity(
                hr_imgs_y.cpu().numpy(), sr_imgs_y.cpu().numpy(), data_range=255.0
            )
            PSNRs.update(psnr, lr_imgs.size(0))
            SSIMs.update(ssim, lr_imgs.size(0))

    # Print average PSNR and SSIM
    print("PSNR - {psnrs.avg:.3f}".format(psnrs=PSNRs))
    print("SSIM - {ssims.avg:.3f}".format(ssims=SSIMs))

print("\n")

In [ ]:
from utils import *
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from datasets import SRDataset
from easydict import EasyDict as edict
import torch

# Configuration
csv_folder = "data"
test_data_names = ["Set5", "Set14", "B100", "Urban100", "valid"]
HR_data_folders = [
    "data/benchmark/Set5/HR",
    "data/benchmark/Set14/HR",
    "data/benchmark/B100/HR",
    "data/benchmark/Urban100/HR",
    "data/DIV2K_valid_HR",
]
LR_data_folders = [
    "data/benchmark/Set5/LR_bicubic/X4",
    "data/benchmark/Set14/LR_bicubic/X4",
    "data/benchmark/B100/LR_bicubic/X4",
    "data/benchmark/Urban100/LR_bicubic/X4",
    "data/DIV2K_valid_LR_bicubic/X4",
]

# Model checkpoints to test
checkpoints = {
    # "SRGAN": "./checkpoint_srgan_final.pth.tar",
    "SRGAN_Best": "./checkpoints/checkpoint_srgan_best.pth.tar",
}

scaling_factor = 4
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("SRGAN Model Evaluation")
print("=" * 80 + "\n")

# Test each checkpoint
for model_name, checkpoint_path in checkpoints.items():
    print(f"\nTesting Model: {model_name}")
    print(f"Checkpoint: {checkpoint_path}\n")

    # Load model
    try:
        srgan_generator = torch.load(checkpoint_path, weights_only=False)["generator"].to(device)
        srgan_generator.eval()
    except:
        print(f"✗ Failed to load {checkpoint_path}, skipping...\n")
        continue

    # Test on each dataset
    for i, dataset_name in enumerate(test_data_names):
        print(f"Evaluating on {dataset_name}...")

        # Create dataset
        config = edict()
        config.csv_folder = csv_folder
        config.HR_data_folder = HR_data_folders[i]
        config.LR_data_folder = LR_data_folders[i]
        config.crop_size = 0
        config.scaling_factor = scaling_factor

        test_dataset = SRDataset(split=dataset_name, config=config)
        test_loader = torch.utils.data.DataLoader(
            test_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True
        )

        PSNRs = AverageMeter()
        SSIMs = AverageMeter()

        with torch.no_grad():
            for j, (lr_imgs, hr_imgs) in enumerate(test_loader):
                lr_imgs = lr_imgs.to(device)
                hr_imgs = hr_imgs.to(device)

                # Convert images
                lr_imgs = convert_image(
                    lr_imgs, source="[0, 1]", target="imagenet-norm", device=device
                )
                hr_imgs = convert_image(
                    hr_imgs, source="[0, 1]", target="[-1, 1]", device=device
                )

                # Generate SR image
                sr_imgs = srgan_generator(lr_imgs)

                # Convert to Y channel for metric calculation
                sr_imgs_y = convert_image(
                    sr_imgs, source="[-1, 1]", target="y-channel", device=device
                ).squeeze(0)
                hr_imgs_y = convert_image(
                    hr_imgs, source="[-1, 1]", target="y-channel", device=device
                ).squeeze(0)

                # Calculate PSNR and SSIM
                psnr = peak_signal_noise_ratio(
                    hr_imgs_y.cpu().numpy(), sr_imgs_y.cpu().numpy(), data_range=255.0
                )
                ssim = structural_similarity(
                    hr_imgs_y.cpu().numpy(), sr_imgs_y.cpu().numpy(), data_range=255.0
                )

                PSNRs.update(psnr, lr_imgs.size(0))
                SSIMs.update(ssim, lr_imgs.size(0))

        # Print results
        print(
            f"  {dataset_name:12s} - PSNR: {PSNRs.avg:.3f} dB | SSIM: {SSIMs.avg:.4f}"
        )

    print()

print("=" * 80)
print("Evaluation Complete!")
print("=" * 80)